## What Is Sliding Window Memory?

### The core idea in one sentence
Instead of sending the **entire** conversation history to the LLM, send only the **last N turns**. As new turns arrive, the oldest ones fall off the back — like a window sliding forward through the conversation.

---

### The mental model — a moving spotlight

Imagine shining a spotlight on a long script. The spotlight only illuminates the last few pages. As new pages are added, the spotlight moves forward — and the earlier pages fall into darkness.

```
Window size = 3 turns

After Turn 3:  [T1] [T2] [T3]          ← window covers all
After Turn 4:       [T2] [T3] [T4]     ← T1 falls off
After Turn 5:            [T3] [T4] [T5] ← T2 falls off
After Turn 6:                 [T4] [T5] [T6] ← T3 falls off
```

The API always receives exactly N turns — no more, no less. Cost is capped. The context window never overflows.

---

### Point 1 — It directly solves Technique 01's cost problem

With buffer memory, Turn 100 sends 100 turns to the API. With a window of size 10, Turn 100 still sends exactly 10 turns — same as Turn 10. Cost is **constant**, not compounding. This is the primary reason this technique exists.

---

### Point 2 — The window is defined in turns, not tokens

A "turn" is one user message plus one assistant response (2 messages). A window of `k=5` keeps the last 5 complete turns — that is 10 messages. This is more intuitive to reason about than raw token counts, and it gives predictable context window utilisation.

---

### Point 3 — Old messages are not deleted — they are evicted from the active window

This is a critical production distinction. When a message "falls off" the window, it should be:
- **Archived** to a persistent store (database, vector store) — not thrown away
- **Potentially retrievable** later via search — not permanently lost

In a naive implementation, old messages are simply discarded. In a production implementation, they are archived. Technique 06 (Vector Store Memory) shows how to search those archives. Sliding Window + Vector Store together form the basis of most production memory systems.

---

### Point 4 — The critical trade-off: recency vs completeness

The buffer remembers everything. The window remembers only the recent past. If a user tells FinCoach their salary in Turn 1 and asks about it in Turn 15 with a window of 10 — the salary is gone from the active context. The agent will ask again, frustrating the user.

This is the fundamental limitation of sliding window memory: **it solves the cost problem but introduces a forgetting problem**.

---

### Point 5 — Choosing the right window size is a production engineering decision

Too small: the agent forgets important facts, frustrates users, gives inconsistent advice.
Too large: approaches buffer memory's cost problem, defeats the purpose.

The right window size depends on:
- Average conversation length in your domain
- How far back the model typically needs to look
- Your token budget per session
- Whether you have a complementary long-term retrieval layer

For FinCoach: a window of 8–10 turns covers the active part of a financial planning session. Critical facts like salary and risk profile should be stored in a separate entity profile (Technique 07) — not left to chance in the window.

---

## FinCoach Example

```
Window size = 3 turns

Turn 1: User: "My salary is ₹1,20,000"        → in window
Turn 2: User: "My expenses are ₹60,000"        → in window
Turn 3: User: "I have an FD of ₹50,000"        → in window

Turn 4: User: "What about SIPs?"
        → Turn 1 EVICTED. Salary is now gone from context.
        → Agent no longer knows the user's income.
        → If it gives investment advice, it must assume or ask again.

Turn 5: User: "How much should I invest based on my salary?"
        → Agent: "Could you remind me of your monthly salary?"
        → User is frustrated. They told you this in Turn 1.
```

This is the forgetting problem. The solution — combining sliding window with entity memory or vector retrieval — comes in later techniques.

---

## Trade-offs

| | |
|---|---|
| ✅ Fixed, predictable token cost per turn | ❌ Loses early context when window moves forward |
| ✅ Context window never overflows | ❌ Agent may forget important facts from earlier turns |
| ✅ Simple to implement and reason about | ❌ Window size is a tuning parameter with no universal answer |
| ✅ Good recent-turn coherence | ❌ No long-term memory across sessions |

## Production Verdict

> **Viable as the short-term layer of a larger memory system. Not sufficient alone.**
>
> Sliding window is almost always present in production systems — but almost never alone. It is paired with a long-term retrieval layer (vector store, knowledge graph) that catches the facts that slide out of the window. The window handles recency and cost. The retrieval layer handles permanence. Together they cover most production needs.

---

In the previous notebook we saw that Conversation Buffer Memory stores everything. That causes input tokens to grow linearly per turn and cumulative cost to grow quadratically. For most chat applications, that's wasteful. Users rarely reference something they said 30 turns ago.

Imagine a whiteboard that only fits five sticky notes. When you add a sixth, you peel off the oldest one and toss it. Sliding Window Memory works the same way. It keeps a fixed-size window of the most recent messages and discards everything older. The last k messages are always available, but older messages are gone.

### Key Concepts

**Window size k***: The maximum number of messages retained. A larger *k means better recall but higher token cost.

FIFO eviction: FIFO stands for "first-in, first-out." When len(messages) > k, the oldest message drops from the front. The window "slides" forward.

collections.deque: Python's double-ended queue with a maxlen parameter. It's a natural fit: appending to a full deque automatically discards the oldest item.

Recency bias: By design, the agent remembers recent context and forgets older context. This is a feature, not a bug, when recent context is what matters most.

Message-count vs. turn-count windows: A message window of k counts each message individually (user or assistant). A turn window of k keeps the last k user-assistant pairs intact. Turn-count windows avoid orphaned messages (a response without its question).

Constant token cost: Unlike buffer memory, per-turn token cost is bounded by the window size. This gives you predictable latency and cost.

### Mermaid source sequenceDiagram

    participant U as User
    participant W as Sliding Window (k=4 messages)
    participant L as LLM (Claude)

    Note over W: Window: []

    U->>W: "Hi, I'm Alice" (msg 1)
    W->>L: [msg1]
    L-->>W: "Hello Alice!" (msg 2)
    Note over W: Window: [msg1, msg2]

    U->>W: "I like Python" (msg 3)
    W->>L: [msg1, msg2, msg3]
    L-->>W: "Python is great!" (msg 4)
    Note over W: Window: [msg1, msg2, msg3, msg4] - FULL

    U->>W: "What's my name?" (msg 5)
    Note over W: ⚠ Window full → drop msg1
    W->>W: Evict msg1, append msg5
    W->>L: [msg2, msg3, msg4, msg5]
    L-->>W: "Your name is Alice." (msg 6)
    Note over W: Drop msg2 → Window: [msg3, msg4, msg5, msg6]

    U->>W: "What language do I like?"
    Note over W: msg3 ("I like Python") still
in window → agent remembers
    W->>L: [msg4, msg5, msg6, msg7]
    L-->>W: "You said you like Python!"
    Note over W: Eventually msg3 slides out
and that fact is forgotten

In [1]:
# Install required packages.
# 'openai'   — OpenAI SDK for GPT-4o API calls.
# 'tiktoken' — GPT-4o's exact tokeniser for token counting.
!pip install openai tiktoken --quiet

In [2]:
# --- Standard library ---
import json                              # For serialising sessions to JSON.
import os                                # For reading environment variables.
import time                              # For rate-limit delays between API calls.
from collections import deque            # The core data structure for the sliding window.
                                         # deque (double-ended queue) supports O(1) append
                                         # and O(1) popleft — perfect for a sliding window.
from datetime import datetime, timezone  # UTC timestamps, Python 3.12+ compatible.
from typing import List, Dict, Optional  # Type hints for clarity and IDE support.
from dataclasses import dataclass, field, asdict  # Clean data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.

In [3]:
# --- API Key Setup ---
# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    # google.colab is only available in Colab — this import fails locally.
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    # Fetch the key from Colab's encrypted secrets vault.
    print("Key loaded from Colab Secrets.")
except Exception:
    # Option B: Environment variable (for local Jupyter / VS Code).
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

# Validate the key looks correct before we try to use it.
assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets or environment."

# Create the OpenAI client — the single object used for all API calls.
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Model and tokeniser constants.
MODEL = "gpt-4o"
# gpt-4o: GPT-4o — OpenAI's flagship model, 128k context window.

TOKENISER = tiktoken.get_encoding("o200k_base")
# o200k_base is the exact tokeniser used by gpt-4o.
# Using the correct tokeniser means our token counts are precise, not approximate.

print(f"Client ready. Model: {MODEL}")

Key loaded from environment variable.
Client ready. Model: gpt-4o


---
## Part 1 — The Message Data Model

Same as Technique 01 — the atomic unit of the conversation. Unchanged.

In [4]:
@dataclass
class Message:
    """
    A single conversation message — role, content, and timestamp.
    Identical to Technique 01. The data model does not change between techniques.
    """

    role: str
    # 'user' or 'assistant' — who sent this message.
    # The system prompt is handled separately, not stored here.

    content: str
    # The text of the message.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set to now when the Message is created.
    # timezone.utc is the Python 3.12+ compatible way to get UTC time.

    def to_api_format(self) -> Dict[str, str]:
        """
        Strip the timestamp and return only what the OpenAI API accepts.
        """
        return {"role": self.role, "content": self.content}
        # The API only accepts 'role' and 'content' — timestamp would cause a 422 error.

print("Message dataclass defined.")

Message dataclass defined.


---
## Part 2 — The SlidingWindowMemory Class

### Why `deque` instead of a `list`

In Technique 01 we used a plain Python `list`. For a sliding window, we use a `deque` (double-ended queue) with a `maxlen` parameter.

When a `deque` with `maxlen=N` receives a new item and is already full, **it automatically drops the oldest item from the left**. This is exactly the sliding window behaviour we need — built into the data structure itself, with no manual eviction logic required.

```python
from collections import deque
d = deque(maxlen=3)
d.append('T1')  → deque(['T1'])
d.append('T2')  → deque(['T1', 'T2'])
d.append('T3')  → deque(['T1', 'T2', 'T3'])  ← full
d.append('T4')  → deque(['T2', 'T3', 'T4'])  ← T1 auto-evicted
d.append('T5')  → deque(['T3', 'T4', 'T5'])  ← T2 auto-evicted
```

In [5]:
class SlidingWindowMemory:
    """
    Keeps only the last `window_size` conversation turns in the active context.
    Older turns are evicted automatically as new ones arrive.
    Evicted turns are archived to a separate list for potential later retrieval.

    Key difference from ConversationBufferMemory:
    - Buffer:  GROWS unboundedly. Turn 100 sends 100 turns.
    - Window:  FIXED size.        Turn 100 sends window_size turns.
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,          # Unique ID for this session.
        user_id: str,             # ID of the user — for multi-tenant isolation.
        system_prompt: str,       # Agent instructions — FinCoach's persona.
        window_size: int = 10,    # Number of TURNS to keep (1 turn = 2 messages).
    ):
        self.session_id = session_id
        # Identifies this session in logs and the persistence store.

        self.user_id = user_id
        # Scopes this buffer to one user — prevents cross-user memory leakage.

        self.system_prompt = system_prompt
        # FinCoach's persona and rules — prepended to every API call.

        self.window_size = window_size
        # How many TURNS to keep in the active window.
        # window_size=10 means 10 user messages + 10 assistant responses = 20 messages maximum.

        self._window: deque = deque(maxlen=window_size * 2)
        # The active sliding window — a deque of Message objects.
        # maxlen = window_size * 2 because each turn has 2 messages (user + assistant).
        # When the deque is full and a new message is appended, the OLDEST message
        # is automatically dropped from the LEFT side — this IS the sliding window.
        # No manual eviction code needed — deque handles it.

        self._archive: List[Message] = []
        # All messages ever in this session — including ones evicted from the window.
        # This is the production-critical addition: evicted messages are NOT lost.
        # They are archived here and could be searched via a vector store (Technique 06)
        # or summarised into long-term memory (Technique 03/04).
        # In production: persist this to a database, not just in RAM.

        self._total_turns = 0
        # Count of ALL turns ever in this session — not just what's in the window.
        # Useful for analytics, overflow policies, and debugging.

        self.created_at = datetime.now(timezone.utc).isoformat()
        # Session creation timestamp — for audit trail.

        print(f"[SlidingWindow] Initialised — session: {self.session_id}, "
              f"user: {self.user_id}, window: {self.window_size} turns "
              f"({self.window_size * 2} messages max)")

    # ------------------------------------------------------------------
    # CORE WINDOW OPERATIONS
    # ------------------------------------------------------------------

    def add_message(self, role: str, content: str) -> None:
        """
        Add a new message to the window.
        If the window is full, the oldest message is automatically evicted
        to the archive by the deque's maxlen mechanism.
        Every message is also recorded in the full archive.
        """

        if role not in ("user", "assistant"):
            # Guard: only user and assistant messages go in the conversation window.
            # System prompt is managed via self.system_prompt, not add_message().
            raise ValueError(f"Invalid role '{role}'. Use 'user' or 'assistant'.")

        new_message = Message(role=role, content=content)
        # Create the Message object with an auto-set UTC timestamp.

        # Before appending, check if the window is about to evict a message.
        # If the deque is at capacity, appending will drop the leftmost item.
        # We capture that outgoing message BEFORE it disappears.
        if len(self._window) == self._window.maxlen:
            # The window is full — appending will auto-evict the oldest message.
            evicted_message = self._window[0]
            # self._window[0] is the oldest message (leftmost item in the deque).
            # We grab a reference to it now, before deque drops it.
            # Note: the archive already contains this message (we added it below
            # when it first arrived) — so no additional archiving is needed here.
            # This log line is just for visibility during the demo.
            print(f"  [EVICT] Sliding window evicting: [{evicted_message.role}] "
                  f"'{evicted_message.content[:45]}...' → archived")

        self._window.append(new_message)
        # Append to the deque. If deque is at maxlen, it auto-drops the leftmost
        # item — the sliding window behaviour happens right here, invisibly.

        self._archive.append(new_message)
        # ALWAYS append to the archive too.
        # The archive is the full session record — it never evicts anything.
        # In production: also write to a database here for durability.

        # Count complete turns (only increment on assistant messages,
        # which marks the completion of a user+assistant turn pair).
        if role == "assistant":
            self._total_turns += 1
            # Increment the turn counter each time an assistant message is added,
            # because that signals a complete turn (user asked, assistant answered).

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Return the current window contents in the format the OpenAI API expects.
        System prompt is prepended as the first message — always.
        Only the last window_size turns are included — not the full history.
        """
        system_message = {"role": "system", "content": self.system_prompt}
        # Build the system message — FinCoach's persona, always first.

        window_messages = [msg.to_api_format() for msg in self._window]
        # Convert the deque of Message objects to a list of plain dicts.
        # list(self._window) preserves the chronological order (oldest first).
        # to_api_format() strips the timestamp, leaving only role and content.

        return [system_message] + window_messages
        # Combine: system message first, then the windowed conversation.
        # This is the ONLY thing sent to the API — not the full archive.

    def get_window_turn_count(self) -> int:
        """
        Return the number of complete turns currently visible in the active window.
        This is at most window_size — never more.
        """
        return len(self._window) // 2
        # Integer division: 8 messages ÷ 2 = 4 complete turns in the window.

    def get_archive_turn_count(self) -> int:
        """
        Return the total number of turns ever in this session,
        including those evicted from the window.
        """
        return self._total_turns
        # Tracks every completed turn, not just those in the window.

    def get_evicted_count(self) -> int:
        """
        Return how many messages have been evicted from the window so far.
        Evicted = total messages ever - messages currently in window.
        """
        return len(self._archive) - len(self._window)
        # Archive holds everything. Window holds the recent slice.
        # The difference is how many messages have slid out of view.

    # ------------------------------------------------------------------
    # TOKEN COUNTING
    # ------------------------------------------------------------------

    def get_window_tokens(self) -> int:
        """
        Count tokens currently in the active window (conversation messages only).
        This is the variable part of the API call cost — bounded by window_size.
        """
        return sum(len(TOKENISER.encode(msg.content)) for msg in self._window)
        # Tokenise each message in the window and sum the counts.
        # With a fixed window size, this number is bounded — it never grows past
        # approximately window_size * avg_tokens_per_turn.

    def get_system_prompt_tokens(self) -> int:
        """
        Count tokens in the system prompt.
        This is the fixed overhead on every API call — same every turn.
        """
        return len(TOKENISER.encode(self.system_prompt))
        # Constant per session — does not change as turns accumulate.

    def get_total_api_tokens(self) -> int:
        """
        Estimate total input tokens for the NEXT API call.
        = system prompt tokens + window tokens.
        With a fixed window, this is bounded and predictable.
        """
        return self.get_system_prompt_tokens() + self.get_window_tokens()
        # The number an engineer uses to confirm the window size is within budget.

    # ------------------------------------------------------------------
    # PERSISTENCE
    # ------------------------------------------------------------------

    def persist(self, filepath: str) -> None:
        """
        Save the full session state to a JSON file.
        Saves BOTH the current window AND the full archive.
        In production: write both to a database.
        The window is the active short-term state.
        The archive is the long-term session record.
        """

        session_record = {
            "schema_version": "1.0",
            # Version the schema for forward compatibility.

            "technique": "sliding_window",
            # Tag which technique produced this record — useful when mixing techniques.

            "session_id": self.session_id,
            "user_id": self.user_id,
            "model": MODEL,
            "created_at": self.created_at,
            "saved_at": datetime.now(timezone.utc).isoformat(),

            "config": {
                "window_size": self.window_size,
                # Save the window size so it is restored correctly on load.
            },

            "stats": {
                "total_turns": self._total_turns,
                # Total turns ever — not just what's in the window.
                "window_turns": self.get_window_turn_count(),
                # Turns currently visible in the active window.
                "archive_messages": len(self._archive),
                # Total messages ever in this session.
                "evicted_messages": self.get_evicted_count(),
                # How many messages have slid out of the window.
                "window_tokens": self.get_window_tokens(),
                # Tokens in the current window — the active API cost.
                "system_tokens": self.get_system_prompt_tokens(),
                # Constant overhead per call.
            },

            "window": [asdict(msg) for msg in self._window],
            # The current active window — what the agent sees right now.
            # asdict() converts each Message dataclass to a plain dict.

            "archive": [asdict(msg) for msg in self._archive],
            # The full session history — including evicted messages.
            # This is what a vector store or summary layer would process.
        }

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(session_record, f, indent=2, ensure_ascii=False)
            # indent=2        : human-readable formatting for debugging.
            # ensure_ascii=False : preserves ₹ and other non-ASCII characters.

        print(f"[SlidingWindow] Persisted — "
              f"{self._total_turns} total turns, "
              f"{self.get_window_turn_count()} in window, "
              f"{self.get_evicted_count()} evicted to archive")

    @classmethod
    def load(cls, filepath: str) -> "SlidingWindowMemory":
        """
        Restore a SlidingWindowMemory from a saved JSON file.
        Restores both the window and the archive.
        @classmethod: called as SlidingWindowMemory.load(filepath).
        """

        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)
            # Parse the JSON file back into a Python dict.

        if record.get("schema_version") != "1.0":
            # Reject unknown schema versions — prevents silent data corruption.
            raise ValueError(f"Unknown schema version: {record.get('schema_version')}")

        mem = cls(
            session_id=record["session_id"],
            user_id=record["user_id"],
            system_prompt="[LOADED — set system_prompt after loading]",
            # System prompt is agent config, not user data — not stored in the file.
            # Caller must set it: loaded_mem.system_prompt = FINCOACH_SYSTEM_PROMPT
            window_size=record["config"]["window_size"],
            # Restore the original window size — must match to avoid re-eviction.
        )

        def msgs_from_list(msg_list: list) -> List[Message]:
            # Helper: convert a list of message dicts back to Message objects.
            return [
                Message(
                    role=m["role"],
                    content=m["content"],
                    timestamp=m["timestamp"],
                )
                for m in msg_list
            ]

        # Restore the archive — the full session history.
        mem._archive = msgs_from_list(record["archive"])
        # This restores all messages ever, including those evicted from the window.

        # Restore the window — only the recent slice.
        # We cannot just assign the list; we must re-populate the deque
        # to respect the maxlen constraint.
        for msg in msgs_from_list(record["window"]):
            mem._window.append(msg)
            # append() respects the deque's maxlen — safe to use here.

        mem._total_turns = record["stats"]["total_turns"]
        # Restore the total turn count — keeps analytics accurate.

        mem.created_at = record["created_at"]
        # Restore original creation time.

        print(f"[SlidingWindow] Loaded — "
              f"{mem._total_turns} total turns, "
              f"{mem.get_window_turn_count()} in window, "
              f"{mem.get_evicted_count()} in archive")

        return mem

    # ------------------------------------------------------------------
    # DIAGNOSTICS
    # ------------------------------------------------------------------

    def print_stats(self) -> None:
        """
        Print a summary of current window state.
        Shows both what the model CAN see (window) and what has been evicted (archive).
        """
        w_tokens  = self.get_window_tokens()
        sys_tokens = self.get_system_prompt_tokens()
        evicted   = self.get_evicted_count()

        print(f"\n{'='*60}")
        print(f"  Sliding Window Stats — Session: {self.session_id}")
        print(f"{'='*60}")
        print(f"  Window size          : {self.window_size} turns ({self.window_size*2} msgs max)")
        print(f"  Turns in window      : {self.get_window_turn_count()} / {self.window_size}")
        print(f"  Total turns (all)    : {self._total_turns}")
        print(f"  Messages in window   : {len(self._window)}")
        print(f"  Messages evicted     : {evicted}")
        # evicted > 0 means the window has already slid past earlier turns.
        print(f"  Archive total        : {len(self._archive)} messages")
        print(f"  Window tokens        : {w_tokens:,}")
        print(f"  System prompt tokens : {sys_tokens:,}")
        print(f"  Next API call input  : ~{w_tokens + sys_tokens:,} tokens")
        # This number stays BOUNDED regardless of how long the session runs.
        # That is the entire point of the sliding window.

        # Visual fill bar for the window.
        fill = self.get_window_turn_count() / self.window_size if self.window_size > 0 else 0
        bar  = "█" * int(30 * fill) + "░" * (30 - int(30 * fill))
        print(f"  Window fill : [{bar}] {fill:.0%}")
        print(f"{'='*60}\n")


print("SlidingWindowMemory class defined.")

SlidingWindowMemory class defined.


---
## Part 3 — The FinCoach Agent with Sliding Window

In [6]:
# FinCoach's system prompt — same as Technique 01.
# Consistent across all 30 techniques — the agent's persona does not change.
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- Flag when you are making assumptions due to missing information.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions.

Use all context in the conversation history to provide personalised, consistent advice."""


def chat(
    memory: SlidingWindowMemory,  # The sliding window memory instance for this session.
    user_message: str,             # The user's current message.
    verbose: bool = True           # Print token stats and window state if True.
) -> str:
    """
    Send one user message to FinCoach via GPT-4o. Return the assistant's reply.
    Identical flow to Technique 01 — only the memory object changes.
    """

    # STEP 1 — Add the user's message to the sliding window.
    memory.add_message(role="user", content=user_message)
    # If the window is full, the oldest message is automatically evicted to the archive.
    # No overflow error needed here — the window never exceeds its size.

    # STEP 2 — Call the OpenAI API with ONLY the windowed context.
    response = client.chat.completions.create(
        model=MODEL,
        # GPT-4o.

        max_tokens=1024,
        # Cap the response length.

        temperature=0.7,
        # Balanced between consistency and naturalness for financial advice.

        messages=memory.get_messages_for_api(),
        # THIS IS THE KEY DIFFERENCE FROM TECHNIQUE 01.
        # get_messages_for_api() returns: [system] + [last N turns only]
        # NOT the full history — just the recent window.
        # The model has no access to turns that have slid out of the window.
    )

    # STEP 3 — Extract the response text.
    assistant_reply = response.choices[0].message.content
    # response.choices[0].message.content — OpenAI response text extraction.
    # Same pattern as Technique 01.

    # STEP 4 — Add the assistant's reply to the window.
    memory.add_message(role="assistant", content=assistant_reply)
    # This completes the turn. Both the user message and assistant reply
    # are now in the window and archive. If the window was already full,
    # this appending will evict one more old message.

    if verbose:
        print(f"[Turn {memory._total_turns}] "
              f"API — prompt: {response.usage.prompt_tokens} tokens, "
              f"completion: {response.usage.completion_tokens} tokens | "
              f"Window: {memory.get_window_turn_count()}/{memory.window_size} turns, "
              f"evicted: {memory.get_evicted_count()} msgs")
        # Watch the 'evicted' count grow as the window slides forward.
        # Watch the 'prompt' tokens stay STABLE once the window is full.

    return assistant_reply


print("FinCoach chat() function defined.")

FinCoach chat() function defined.


---
## Part 4 — Demo: Watching the Window Slide

We use a small `window_size=3` so the sliding behaviour is immediately visible.
We run 6 turns — so turns 1, 2, and 3 will be evicted as turns 4, 5, 6 arrive.

**Watch for:**
- The `[EVICT]` lines in the output — exactly when old turns fall off
- The `prompt` tokens staying roughly constant once the window is full
- Turn 6 asking about the salary from Turn 1 — and FinCoach not knowing it

In [ ]:
sliding_memory = SlidingWindowMemory(
    session_id="session_sw_forgetting_demo",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    window_size=3,
)

demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Turn 1: salary shared here.
    # This WILL be evicted at Turn 4.

    "My monthly expenses are about ₹60,000 for rent, food, and transport.",
    # Turn 2: expenses shared here.
    # This WILL be evicted at Turn 5.

    "I have an FD of ₹50,000 that matures in 3 months. I am risk-averse.",
    # Turn 3: FD and risk profile.
    # This WILL be evicted at Turn 6.

    "What is a Systematic Investment Plan and how does it work?",
    # Turn 4: GENERIC EDUCATIONAL QUESTION — no reason for the AI
    # to mention the user's salary in its answer.
    # After this turn: Turn 1 (salary) is EVICTED from the window.
    # The AI answer here will be a general definition of SIPs —
    # it will NOT echo ₹1,20,000 back into the context.

    "What are the different types of mutual funds available in India?",
    # Turn 5: ANOTHER GENERIC QUESTION — no personal finance context needed.
    # After this turn: Turn 2 (expenses) is EVICTED from the window.
    # The AI answer here will list fund categories generically —
    # salary will NOT appear in the response.

    "What is my exact monthly take-home salary that I told you at the start?",
    # Turn 6: DIRECT RECALL QUESTION — explicitly asks for the salary.
    # At this point:
    #   - Turn 1 (where salary was shared) → EVICTED at Turn 4
    #   - Turn 2 AI response               → EVICTED at Turn 5
    #   - Turn 5 AI response               → generic fund list, no salary
    #   - Salary has NOT been echoed back anywhere in the current window
    #   - The model CANNOT answer this correctly.
    # Expected output: model says it does not have that information,
    # OR hallucinates a number — both are the forgetting failure we want.
]

print("CLEAN FORGETTING DEMO — Sliding Window Memory")
print("Strategy: Generic questions in Turns 4-5 prevent salary echo")
print("Expected: Turn 6 fails — model cannot recall salary")
print("=" * 65)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=sliding_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
print("Demo complete.")
sliding_memory.print_stats()

# ─────────────────────────────────────────────────────────────────────────────
# WHAT TO LOOK FOR IN THE OUTPUT:
#
# Turn 4 AI response: Should explain SIPs generically — no mention of ₹1,20,000
# Turn 5 AI response: Should list fund types generically — no mention of salary
# Turn 6 AI response: Should say one of:
#   (a) "I don't have your salary in our current conversation" ← clean failure
#   (b) Gives a wrong/hallucinated number                      ← also a failure
#   (c) Says "you haven't shared your salary with me"          ← clean failure
#


[SlidingWindow] Initialised — session: session_sw_forgetting_demo, user: chiru_001, window: 3 turns (6 messages max)
CLEAN FORGETTING DEMO — Sliding Window Memory
Strategy: Generic questions in Turns 4-5 prevent salary echo
Expected: Turn 6 fails — model cannot recall salary

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
[Turn 1] API — prompt: 162 tokens, completion: 88 tokens | Window: 1/3 turns, evicted: 0 msgs
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, you have a solid foundation for building a strong financial plan. To get started, it's important to consider your financial goals. Are you looking to save for specific goals like buying a home, retirement, or a vacation? Understanding your expenses and current savings will also help in creating a budget. Let me know if you need guidance on any of these areas!

--- Turn 2 ---
User: My monthly expenses are about ₹60,000 for rent, food, and transport.
[Turn 2] API — prompt: 275 token

---
## Part 5 — Cost Comparison: Buffer vs Sliding Window

No API calls — pure token arithmetic. This is the table that makes the case for sliding window in a production cost review.

In [11]:
def compare_buffer_vs_window(
    turns: List[str],
    window_size: int = 5,
    avg_response_tokens: int = 80
) -> None:
    """
    Compare token cost per turn for buffer memory vs sliding window memory.
    No API calls — pure arithmetic using tiktoken.

    Args:
        turns:                Simulated user messages.
        window_size:          Number of turns to keep in the sliding window.
        avg_response_tokens:  Average tokens per assistant response.
    """

    INPUT_COST  = 2.50  / 1_000_000  # GPT-4o input cost per token.
    OUTPUT_COST = 10.00 / 1_000_000  # GPT-4o output cost per token.

    system_tokens = len(TOKENISER.encode(FINCOACH_SYSTEM_PROMPT))
    # System prompt tokens — constant overhead on both approaches.

    # Track running state for both approaches.
    buffer_history_tokens  = 0  # Grows every turn — never resets.
    window_history_tokens  = 0  # Bounded — resets to window_size * avg_turn_tokens.
    window_turns_deque: deque = deque(maxlen=window_size * 2)
    # Simulate the window's deque with the same maxlen logic.

    buffer_total_cost = 0.0   # Cumulative API cost using buffer approach.
    window_total_cost = 0.0   # Cumulative API cost using window approach.

    print(f"System prompt: {system_tokens} tokens (constant overhead)")
    print(f"Window size  : {window_size} turns")
    print()
    print(f"{'Turn':>4} | {'Buffer Prompt':>13} | {'Window Prompt':>13} | "
          f"{'Buffer Cost':>11} | {'Window Cost':>11} | {'Savings':>9}")
    print("-" * 70)

    for i, user_msg in enumerate(turns, start=1):
        user_tokens = len(TOKENISER.encode(user_msg))
        # Tokens in this user's message.

        # --- Buffer approach ---
        buffer_history_tokens += user_tokens
        # Add this user message to the buffer's running total.
        buffer_prompt = system_tokens + buffer_history_tokens
        # Buffer sends: system + ALL prior messages + this user message.
        buffer_call_cost = buffer_prompt * INPUT_COST + avg_response_tokens * OUTPUT_COST
        buffer_total_cost += buffer_call_cost
        buffer_history_tokens += avg_response_tokens
        # After the call, the assistant's response is also added to the buffer.

        # --- Window approach ---
        window_turns_deque.append(user_tokens)
        # Add user message tokens to the window deque.
        window_turns_deque.append(avg_response_tokens)
        # Add expected response tokens — they'll be in the window after the call.
        window_history_tokens = sum(window_turns_deque)
        # Re-sum the window contents — this reflects the maxlen eviction.
        window_prompt = system_tokens + window_history_tokens
        # Window sends: system + only the last window_size turns.
        window_call_cost = window_prompt * INPUT_COST + avg_response_tokens * OUTPUT_COST
        window_total_cost += window_call_cost

        # Savings this turn.
        turn_savings = buffer_prompt - window_prompt
        # How many fewer tokens the window sent vs the buffer this turn.

        print(f"{i:>4} | {buffer_prompt:>13,} | {window_prompt:>13,} | "
              f"${buffer_call_cost:>10.5f} | ${window_call_cost:>10.5f} | "
              f"{turn_savings:>7,} tkns")

    print("-" * 70)
    total_savings_pct = (1 - window_total_cost / buffer_total_cost) * 100
    # Percentage reduction in total API cost using window vs buffer.

    print(f"\nTotal cost — Buffer : ${buffer_total_cost:.5f}")
    print(f"Total cost — Window : ${window_total_cost:.5f}")
    print(f"Total savings       : {total_savings_pct:.1f}% cheaper with sliding window")
    print(f"\nKey insight: after Turn {window_size}, the window's per-turn prompt cost")
    print(f"stays roughly CONSTANT. The buffer's keeps growing every turn.")


# Simulate a 15-turn FinCoach session.
simulated_turns = [
    "Hi, I'm Chiru. My monthly salary is ₹1,20,000.",
    "My monthly expenses are ₹60,000.",
    "I have an FD of ₹50,000 maturing in 3 months.",
    "I'm risk-averse and prefer stable returns.",
    "Should I invest in mutual funds or fixed deposits?",
    "What about SIPs? How much should I invest monthly?",
    "Is ₹10,000 per month in SIPs realistic given my expenses?",
    "Which fund categories suit a conservative investor?",
    "How long should I stay invested to see meaningful returns?",
    "Can you give me a complete monthly budget breakdown?",
    "What is the right emergency fund size for someone like me?",
    "Should I use a part of my FD to build the emergency fund?",
    "What tax-saving instruments should I look at under Section 80C?",
    "Is ELSS better than PPF for someone my age?",
    "Can you summarise the action plan we've discussed?",
]

compare_buffer_vs_window(simulated_turns, window_size=5)

System prompt: 132 tokens (constant overhead)
Window size  : 5 turns

Turn | Buffer Prompt | Window Prompt | Buffer Cost | Window Cost |   Savings
----------------------------------------------------------------------
   1 |           149 |           229 | $   0.00117 | $   0.00137 |     -80 tkns
   2 |           238 |           318 | $   0.00139 | $   0.00160 |     -80 tkns
   3 |           334 |           414 | $   0.00164 | $   0.00184 |     -80 tkns
   4 |           423 |           503 | $   0.00186 | $   0.00206 |     -80 tkns
   5 |           513 |           593 | $   0.00208 | $   0.00228 |     -80 tkns
   6 |           605 |           588 | $   0.00231 | $   0.00227 |      17 tkns
   7 |           700 |           594 | $   0.00255 | $   0.00229 |     106 tkns
   8 |           788 |           586 | $   0.00277 | $   0.00227 |     202 tkns
   9 |           879 |           588 | $   0.00300 | $   0.00227 |     291 tkns
  10 |           969 |           588 | $   0.00322 | $   0.002

---
## Part 6 — The Forgetting Problem: Demonstrating What Gets Lost

In [13]:
def demonstrate_forgetting(
    window_size: int,
    fact_turn: int,
    query_turn: int
) -> None:
    """
    Show exactly whether a fact shared at `fact_turn` is still
    visible to the model at `query_turn`, given a window of `window_size`.

    This is the calculation every engineer should run when choosing
    their window size for a new domain.

    Args:
        window_size: Number of turns kept in the window.
        fact_turn:   The turn number when the important fact was shared.
        query_turn:  The turn number when the model needs to use that fact.
    """

    turns_since_fact = query_turn - fact_turn
    # How many turns have passed since the fact was shared.

    fact_still_visible = turns_since_fact < window_size
    # The fact is visible only if it's within the last window_size turns.
    # If turns_since_fact >= window_size, the fact has been evicted.

    status = "✅ IN WINDOW — model CAN see this fact" if fact_still_visible \
             else "❌ EVICTED — model CANNOT see this fact — may ask again"

    print(f"  Fact shared at turn {fact_turn:>3}, "
          f"queried at turn {query_turn:>3}, "
          f"window size {window_size:>3}: {status}")


print("=" * 70)
print("FinCoach — Forgetting Analysis")
print("Key facts shared and when they might be needed")
print("=" * 70)

# Scenario 1: window_size = 5
print("\nWindow size = 5 turns")
demonstrate_forgetting(window_size=5, fact_turn=1,  query_turn=6)   # Salary asked late
demonstrate_forgetting(window_size=5, fact_turn=1,  query_turn=3)   # Salary asked soon
demonstrate_forgetting(window_size=5, fact_turn=3,  query_turn=8)   # Risk profile asked late
demonstrate_forgetting(window_size=5, fact_turn=3,  query_turn=5)   # Risk profile asked soon

# Scenario 2: window_size = 10
print("\nWindow size = 10 turns")
demonstrate_forgetting(window_size=10, fact_turn=1,  query_turn=6)  # Same queries — larger window
demonstrate_forgetting(window_size=10, fact_turn=1,  query_turn=11) # Salary just barely evicted
demonstrate_forgetting(window_size=10, fact_turn=3,  query_turn=8)  # Risk profile — safe
demonstrate_forgetting(window_size=10, fact_turn=3,  query_turn=15) # Risk profile — evicted

# Scenario 3: window_size = 20
print("\nWindow size = 20 turns")
demonstrate_forgetting(window_size=20, fact_turn=1,  query_turn=15) # Salary — safe for long session
demonstrate_forgetting(window_size=20, fact_turn=1,  query_turn=22) # Salary — finally evicted

print("\nConclusion:")
print("  For FinCoach, critical facts (salary, risk profile) are shared in turns 1-3.")
print("  A window_size of 10 keeps them safe for up to ~10 follow-up turns.")
print("  For sessions longer than 10 turns, pair sliding window with an entity")
print("  profile store to preserve these facts permanently.")

FinCoach — Forgetting Analysis
Key facts shared and when they might be needed

Window size = 5 turns
  Fact shared at turn   1, queried at turn   6, window size   5: ❌ EVICTED — model CANNOT see this fact — may ask again
  Fact shared at turn   1, queried at turn   3, window size   5: ✅ IN WINDOW — model CAN see this fact
  Fact shared at turn   3, queried at turn   8, window size   5: ❌ EVICTED — model CANNOT see this fact — may ask again
  Fact shared at turn   3, queried at turn   5, window size   5: ✅ IN WINDOW — model CAN see this fact

Window size = 10 turns
  Fact shared at turn   1, queried at turn   6, window size  10: ✅ IN WINDOW — model CAN see this fact
  Fact shared at turn   1, queried at turn  11, window size  10: ❌ EVICTED — model CANNOT see this fact — may ask again
  Fact shared at turn   3, queried at turn   8, window size  10: ✅ IN WINDOW — model CAN see this fact
  Fact shared at turn   3, queried at turn  15, window size  10: ❌ EVICTED — model CANNOT see this fact

---
## Key Takeaways

**What you built:** A `SlidingWindowMemory` class using a `deque` with `maxlen` to enforce a fixed-size context window, with automatic eviction to an archive, session persistence, and a cost comparison showing exactly how the window caps token costs.

---

### The progression so far

| | Technique 01 — Buffer | Technique 02 — Window |
|---|---|---|
| Token cost | Grows every turn | Fixed after window fills |
| Context overflow risk | High for long sessions | None — size is capped |
| Memory of early facts | Perfect | Lost after window moves |
| Implementation complexity | Minimal | Minimal |
| Production readiness | Not alone | Not alone — needs a retrieval layer |

---

### The three things to remember

1. **The `deque` with `maxlen` is the entire mechanism.** Python's built-in double-ended queue auto-evicts the oldest item when full. The sliding window is not complex logic — it is the right data structure.

2. **Evicted ≠ deleted.** In a production system, messages that leave the window go to an archive — not the bin. That archive is what Technique 06 (vector store) and Technique 07 (entity memory) draw from. The window is the fast lane; the archive is the long-term store.

3. **Window size is a domain decision, not a code decision.** For FinCoach, critical facts are shared in the first 3 turns. A window of 10 keeps them safe for most sessions. For sessions longer than 10 turns, you need a complementary layer — which is exactly what Technique 03 (Summary Memory) provides.

---

### What breaks next — and what Technique 03 fixes

The sliding window caps cost but loses early context. The next question is: **can we preserve the meaning of early turns without paying to keep them verbatim?**

Yes — by summarising them. Technique 03 (Summary Memory) uses the LLM itself to compress old turns into a running summary that stays in context even as the raw messages scroll out of the window. The summary is compact, the recent turns are verbatim, and nothing important is truly lost.

### FAQ

Q: What is Sliding Window Memory in agent memory?

A: Sliding Window Memory keeps only the last K messages in the conversation history and discards older ones using FIFO (first-in, first-out) eviction. This bounds token usage to a predictable maximum regardless of conversation length. For example, with K=10 and roughly 50 tokens per message, you always spend about 500 tokens on history. LangChain implements this as ConversationBufferWindowMemory with a configurable k parameter.

Q: When should I use Sliding Window Memory instead of Token Buffer Memory?

A: Use Sliding Window Memory when you want the simplest possible bounded-memory approach and message lengths are roughly uniform. Token Buffer Memory (technique 05) trims by token count, giving finer control when messages vary in length. If some messages are 10 tokens and others are 500, a message-count window wastes budget or overflows unpredictably. For chatbots with short, consistent turns, the sliding window is easier to reason about and configure.

Q: What are the limits or failure modes of Sliding Window Memory?

A: The main failure mode is context amnesia. Once a message scrolls past the window, it is gone forever. The agent cannot recall early instructions, user preferences stated in turn 1, or prior agreements. If K is too small, the agent loses critical context mid-conversation. If K is too large, you approach the same cost problems as Conversation Buffer Memory. There is no graceful degradation, only a hard cutoff at K messages.

Q: Can I combine Sliding Window Memory with another memory technique?

A: Yes. Pair it with technique 03 (Summary Memory) for a lightweight hybrid. Keep the last K messages verbatim in the window, and maintain a running summary of everything that scrolled out. This is exactly what technique 04 (Summary Buffer Memory) formalizes. You can also combine it with technique 06 (Vector Store Memory) to retrieve relevant older messages on demand while keeping the window for immediate context.

Q: What library or framework can I use to skip the implementation work?

A: LangChain provides ConversationBufferWindowMemory with a k parameter out of the box. LlamaIndex offers ChatMemoryBuffer with a token_limit option that provides similar bounded behavior. For managed solutions, Zep (technique 27) and Mem0 (technique 25) both support configurable retention windows. If you need finer control at the token level rather than message level, see technique 05 (Token Buffer Memory) and LangChain's ConversationTokenBufferMemory.